# BCI Group Comparison — Standalone

Self-contained notebook: discovers sessions from mounted `/data/` assets,
loads raw data via `ddc.main()`, computes all metrics, and produces three figures.
Does **not** depend on any other notebook or pre-generated CSV.

| Figure | Content |
|--------|--------|
| **Fig 1** | Animal performance — hit rate, TTR trajectory, threshold advancement, quartile TTR change |
| **Fig 2** | CN activity and timing — mean level, within-session trajectory, post-reward response, peak-window CN |
| **Fig 3** | CN–behavior relationships — scatter correlations including CN–population spike coupling |

> **Statistics** — Mann-Whitney U on pooled session data (sessions within the same animal
> are not independent; p-values are anti-conservative until group sizes grow).


In [1]:
import os, re, json, sys, subprocess, warnings, traceback
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
from scipy import stats

sys.path.insert(0, '/code')
import extract_scanimage_metadata
import data_dict_create_module_bruker as ddc

warnings.filterwarnings('ignore')
print('Imports OK')


could not import s2p
Imports OK


In [2]:
# ── Group definitions ─────────────────────────────────────────────────────
# Map group label -> list of subject ID strings as they appear in asset names
GROUPS = {
    'Ctrl':  ['820614', '824946', '820615', '855519'],
    'LC-KO': ['857094', '857095', '849680'],
}

# Optional: pin colours per group. Set to None to auto-assign from tab10.
GROUP_COLOR_OVERRIDES = {
    'Ctrl':  '#185FA5',   # blue
    'LC-KO': '#D85A30',   # coral
}

DATE_FILTER      = None   # e.g. '2026-06-04', or None for all dates

# ── Trial windowing ────────────────────────────────────────────────────────
TRIAL_START      = 10     # 0-indexed; skip first N trials (PMT artefact / warm-up)
TRIAL_END_OFFSET = 10     # drop last N trials

# ── Exclusions (applied after windowing) ──────────────────────────────────
MIN_HITS         = 1
MIN_TRIALS       = 10

# ── Analysis parameters ────────────────────────────────────────────────────
HIT_WINDOW   = 10          # moving-avg window for hit rate
CN_SMOOTH    = 10          # smoothing window for CN directionality
BASELINE_SEC = (-5., -4.)  # peri-reward baseline window (seconds)
PLOT_SEC     = (-5.,  5.)  # peri-reward display window (seconds)
WIN_SIZE     = 20          # local-max learning window (trials)

OUT_DIR = Path('/results')
OUT_DIR.mkdir(exist_ok=True)

# ── Build colour maps ──────────────────────────────────────────────────────
_tab10 = plt.cm.tab10.colors
COLORS, MED_COLORS, GROUP_X = {}, {}, {}
for _i, _g in enumerate(GROUPS):
    _base = (GROUP_COLOR_OVERRIDES.get(_g)
             if GROUP_COLOR_OVERRIDES else None)
    if _base is None:
        _base = mcolors.to_hex(_tab10[_i % len(_tab10)])
    COLORS[_g]     = _base
    MED_COLORS[_g] = mcolors.to_hex([c * 0.45 for c in mcolors.to_rgb(_base)])
    GROUP_X[_g]    = _i

JITTER_W    = 0.22
JITTER_SEED = 42

plt.rcParams.update({
    'font.family': 'sans-serif', 'font.size': 10,
    'axes.linewidth': 0.8, 'figure.dpi': 150,
    'savefig.dpi': 300, 'savefig.bbox': 'tight',
})

print('Config OK')
print('Groups:', {g: len(ids) for g, ids in GROUPS.items()})
print('Colors:', COLORS)


Config OK
Groups: {'Ctrl': 4, 'LC-KO': 3}
Colors: {'Ctrl': '#185FA5', 'LC-KO': '#D85A30'}


In [3]:
def _json_ok(line):
    try:
        json.loads(line); return True
    except json.JSONDecodeError:
        return False


def find_pair_in_data(subject, date):
    prefix   = f'single-plane-ophys_{subject}_{date}_'
    attached = sorted(Path('/data').iterdir())
    raws  = [p for p in attached
             if p.name.startswith(prefix) and '_processed_' not in p.name]
    procs = [p for p in attached
             if p not in raws and subject in p.name
             and date in p.name and '_processed_' in p.name]
    if not raws:  raise RuntimeError(f'No raw asset for {subject}/{date}')
    if not procs: raise RuntimeError(f'No processed asset for {subject}/{date}')
    def _ts(p):
        m = re.search(r'_processed_(\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2})', p.name)
        return m.group(1) if m else ''
    proc = sorted(procs, key=_ts, reverse=True)[0]
    if (proc / 'extraction').is_dir():
        proc_root = proc
    else:
        candidates = [c for c in proc.iterdir()
                      if c.is_dir() and (c / 'extraction').is_dir()]
        if not candidates:
            raise RuntimeError(f'No extraction/ in {proc.name}')
        proc_root = candidates[0]
    return raws[0], proc, proc_root


def _best_bci_stem(proc_root):
    trial_json = proc_root / 'motion_correction' / 'trial_locations.json'
    if not trial_json.exists(): return 'bci'
    with open(trial_json) as f:
        trial_locs = json.load(f)
    stem_counts = {}
    for name in trial_locs:
        parts = name.rsplit('_', 1)
        if len(parts) == 2 and re.match(r'bci\d*$', parts[0]):
            stem_counts[parts[0]] = stem_counts.get(parts[0], 0) + 1
    return max(stem_counts, key=stem_counts.get) if stem_counts else 'bci'


def build_workspace(subject, date, target_stem):
    tag       = f'{subject}_{date}_{target_stem}'
    workspace = Path(f'/scratch/grp_{tag}')
    pophys    = workspace / 'pophys'

    raw, _, proc_root = find_pair_in_data(subject, date)
    extraction = proc_root / 'extraction'
    mc         = proc_root / 'motion_correction'

    with open(mc / 'epoch_locations.json') as f:
        epoch_locs = json.load(f)
    with open(mc / 'trial_locations.json') as f:
        trial_locs = json.load(f)

    if target_stem not in epoch_locs:
        base  = target_stem.rstrip('0123456789') or target_stem
        cands = sorted(k for k in epoch_locs if base in k)
        if not cands:
            raise RuntimeError(f'{target_stem!r} not in epoch_locations: {list(epoch_locs)}')
        target_stem = base if base in epoch_locs else cands[0]

    t_start, t_end = epoch_locs[target_stem]
    subprocess.run(['rm', '-rf', str(workspace)], check=False)
    pophys.mkdir(parents=True)

    # Behaviour
    bsrc = raw / 'behavior'; bdst = workspace / 'behavior'
    for src in bsrc.rglob('*'):
        rel = src.relative_to(bsrc); dst = bdst / rel
        if src.is_dir():
            dst.mkdir(parents=True, exist_ok=True)
        elif src.suffix == '.json' and rel.parts[0] == 'SoftwareEvents':
            good = [l for l in src.read_text(errors='replace').splitlines()
                    if l.strip() and _json_ok(l)]
            dst.write_text('\n'.join(good) + '\n')
        elif not dst.exists():
            dst.symlink_to(src)

    # pophys ancillary files
    for f in (raw / 'pophys').iterdir():
        if (f.is_file() and f.suffix not in ('.tif', '.stim')
                and f.name.startswith(f'{target_stem}_')):
            tgt = pophys / f.name
            if not tgt.exists(): tgt.symlink_to(f)

    # Fluorescence / spike arrays
    bci_dir     = pophys / 'suite2p_BCI' / 'plane0'
    bci_dir.mkdir(parents=True)
    frame_slice = slice(t_start, t_end + 1)
    target_tifs = sorted(
        name for name, (s, e) in trial_locs.items()
        if name.startswith(f'{target_stem}_') and s >= t_start and e <= t_end
    )
    fpf = [trial_locs[n][1] - trial_locs[n][0] + 1 for n in target_tifs]

    # ── Key fix: use pre-sliced suite2p_bci/plane0 files when available ────
    # These already contain only the BCI epoch, saving slicing overhead and
    # ensuring spks.npy is always present where suite2p ran successfully.
    _s2p_sub = extraction / 'suite2p_bci' / 'plane0'
    for fname in ['F', 'Fneu', 'spks']:
        dst      = bci_dir / f'{fname}.npy'
        src_pre  = _s2p_sub / f'{fname}.npy'
        src_full = extraction / f'{fname}.npy'
        if src_pre.exists():
            dst.symlink_to(src_pre)
        elif src_full.exists():
            arr = np.load(src_full, mmap_mode='r')
            np.save(dst, arr[:, frame_slice])
            del arr

    for fname in ['stat.npy', 'iscell.npy']:
        src = extraction / fname
        if src.exists():
            dst = bci_dir / fname
            if not dst.exists(): dst.symlink_to(src)

    ops = np.load(extraction / 'ops.npy', allow_pickle=True).tolist()
    ops['frames_per_file'] = fpf
    np.save(bci_dir / 'ops.npy', ops)

    si = extract_scanimage_metadata.extract_scanimage_metadata(
        str(raw / 'pophys' / target_tifs[0]))
    si['siBase']      = {0: target_stem, 1: '', 2: 'spont_pre'}
    si['savefolders'] = {0: target_stem, 1: 'spont', 2: 'spont_post',
                         3: 'spont_pre', 4: 'spont_post'}
    np.save(bci_dir / 'siHeader.npy', si)

    _spks_ok = (bci_dir / 'spks.npy').exists()
    print(f'  workspace: grp_{tag}  ({len(target_tifs)} TIFFs, '
          f'{sum(fpf)} frames, spks={_spks_ok})')
    return pophys


print('Discovery helpers OK')


Discovery helpers OK


In [4]:
def _safe_mean(arr):
    a = np.asarray(arr, dtype=float)
    return float(np.nanmean(a)) if len(a) > 0 else np.nan

def moving_average(arr, window):
    arr = np.asarray(arr, dtype=float)
    out = np.empty(len(arr))
    for i in range(len(arr)):
        out[i] = np.mean(arr[max(0, i - window + 1):i + 1])
    return out

def cn_directionality_metric(trace, smooth_window):
    trace = np.asarray(trace, dtype=float)
    if len(trace) < smooth_window + 2: return np.nan
    nans = np.isnan(trace)
    if nans.all(): return np.nan
    if nans.any():
        x = np.arange(len(trace))
        trace = np.interp(x, x[~nans], trace[~nans])
    sm    = np.convolve(trace, np.ones(smooth_window)/smooth_window, mode='valid')
    deriv = np.diff(sm)
    above = deriv[deriv > 0].sum()
    below = np.abs(deriv[deriv < 0]).sum()
    total = above + below
    return 0.0 if total == 0 else float((above - below) / total)

def peri_reward_trace(cn_trace, reward_frames, fs, baseline_sec, plot_sec):
    n      = len(cn_trace)
    b0, b1 = int(baseline_sec[0]*fs), int(baseline_sec[1]*fs)
    p0, p1 = int(plot_sec[0]*fs),     int(plot_sec[1]*fs)
    n_plot = p1 - p0
    trials = []
    for rf in reward_frames:
        if np.isnan(rf): continue
        rf = int(rf)
        bs, be, ps, pe = rf+b0, rf+b1, rf+p0, rf+p1
        if bs < 0 or be > n or ps < 0 or pe > n: continue
        t = cn_trace[ps:pe] - float(np.mean(cn_trace[bs:be]))
        if len(t) == n_plot: trials.append(t)
    return np.mean(trials, axis=0) if trials else None

def zscore_to_baseline(trace, rs, re):
    mu    = float(np.nanmean(trace[rs:re]))
    sigma = float(np.nanstd(trace[rs:re]))
    return (trace - mu) / (sigma if sigma > 0 else 1.0)

def epoch_switches(thresh):
    k_upper = np.diff(thresh[1, :])
    return np.concatenate(([0],
           np.where((k_upper != 0) & (~np.isnan(k_upper)))[0]))

def group_mwu(data, col):
    arrays = [data.loc[data['group']==g, col].dropna().values for g in GROUPS]
    valid  = [a for a in arrays if len(a) > 1]
    if len(valid) < 2: return np.nan, ''
    try:
        if len(valid) == 2:
            _, p = stats.mannwhitneyu(valid[0], valid[1], alternative='two-sided')
            return p, 'MWU'
        _, p = stats.kruskal(*valid)
        return p, 'K-W'
    except Exception:
        return np.nan, ''

print('Analysis helpers OK')


Analysis helpers OK


In [5]:
def _clean(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(labelsize=9)

def _label(ax, letter):
    ax.text(-0.15, 1.06, letter, transform=ax.transAxes,
            fontsize=13, fontweight='bold', va='top')


def strip_plot(ax, data, col, ref_y=None, ylabel=None, ylim=None):
    rng = np.random.default_rng(JITTER_SEED)
    ns  = {}
    for g in GROUPS:
        vals = data.loc[data['group']==g, col].dropna()
        ns[g] = len(vals)
        if vals.empty: continue
        jit = rng.uniform(-JITTER_W/2, JITTER_W/2, len(vals))
        ax.scatter(GROUP_X[g]+jit, vals.values,
                   color=COLORS[g], alpha=0.50, s=22, linewidths=0, zorder=2)
        ax.hlines(vals.median(), GROUP_X[g]-0.26, GROUP_X[g]+0.26,
                  colors=MED_COLORS[g], linewidths=2.5, zorder=3)
    p, _ = group_mwu(data, col)
    if not np.isnan(p):
        pfmt = f'p = {p:.3f}' if p >= 0.001 else 'p < 0.001'
        ax.text(0.97, 0.97, pfmt, transform=ax.transAxes,
                ha='right', va='top', fontsize=8,
                color='#222' if p < 0.05 else '#aaa')
    if ref_y is not None:
        ax.axhline(ref_y, color='#bbb', lw=1.0, ls='--', zorder=1)
    ax.set_xticks(list(GROUP_X.values()))
    ax.set_xticklabels([f'{g}\n(n={ns.get(g,0)})' for g in GROUPS], fontsize=9)
    ax.set_xlim(-0.6, len(GROUPS)-0.4)
    if ylim:   ax.set_ylim(ylim)
    if ylabel: ax.set_ylabel(ylabel, fontsize=10)
    _clean(ax)


def slope_chart(ax, data, col1, col2, ylabel=None):
    rng = np.random.default_rng(JITTER_SEED)
    for g in GROUPS:
        sub = data.loc[data['group']==g, [col1, col2]].dropna()
        if sub.empty: continue
        jit = rng.uniform(-0.04, 0.04, len(sub))
        for i, (_, row) in enumerate(sub.iterrows()):
            ax.plot([jit[i], 1+jit[i]], [row[col1], row[col2]],
                    color=COLORS[g], alpha=0.22, lw=0.9)
        mu = [sub[col1].mean(), sub[col2].mean()]
        _, pt = stats.ttest_rel(sub[col1], sub[col2])
        ax.plot([0, 1], mu, color=MED_COLORS[g], lw=2.5, zorder=5,
                label=f'{g} (paired p={pt:.3f})')
        ax.scatter([0, 1], mu, color=MED_COLORS[g], s=45, zorder=6)
    gnames = list(GROUPS)
    if len(gnames) >= 2:
        d0 = (data.loc[data['group']==gnames[0],[col1,col2]].dropna()
              .apply(lambda r: r[col2]-r[col1], axis=1))
        d1 = (data.loc[data['group']==gnames[1],[col1,col2]].dropna()
              .apply(lambda r: r[col2]-r[col1], axis=1))
        if len(d0) > 1 and len(d1) > 1:
            _, p_grp = stats.mannwhitneyu(d0, d1, alternative='two-sided')
            ax.text(0.97, 0.03, f'\u0394 group p = {p_grp:.3f}',
                    transform=ax.transAxes, ha='right', va='bottom', fontsize=8,
                    color='#222' if p_grp < 0.05 else '#aaa')
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['First \u2153', 'Last \u2153'], fontsize=9)
    ax.set_xlim(-0.30, 1.30)
    if ylabel: ax.set_ylabel(ylabel, fontsize=10)
    ax.legend(fontsize=7.5, frameon=False, loc='lower right')
    _clean(ax)


def scatter_corr(ax, data, x_col, y_col,
                 xlabel=None, ylabel=None,
                 ref_x=None, ref_y=None,
                 xlim=None, ylim=None):
    for g in GROUPS:
        sub = data.loc[data['group']==g, [x_col, y_col]].dropna()
        if sub.empty: continue
        ax.scatter(sub[x_col], sub[y_col],
                   color=COLORS[g], alpha=0.60, s=28,
                   linewidths=0, label=g, zorder=2)
        if len(sub) > 2:
            m, b = np.polyfit(sub[x_col], sub[y_col], 1)
            xs = np.linspace(sub[x_col].min(), sub[x_col].max(), 60)
            ax.plot(xs, m*xs+b, color=MED_COLORS[g], lw=1.5, alpha=0.9)
    both = data[[x_col, y_col]].dropna()
    if len(both) > 4:
        r, p = stats.spearmanr(both[x_col], both[y_col])
        pfmt = f'p = {p:.3f}' if p >= 0.001 else 'p < 0.001'
        ax.text(0.04, 0.97, f'\u03c1 = {r:+.2f}, {pfmt}',
                transform=ax.transAxes, ha='left', va='top',
                fontsize=8, color='#333')
    if ref_x is not None: ax.axvline(ref_x, color='#ccc', lw=0.8, ls='--')
    if ref_y is not None: ax.axhline(ref_y, color='#ccc', lw=0.8, ls='--')
    if xlabel: ax.set_xlabel(xlabel, fontsize=10)
    if ylabel: ax.set_ylabel(ylabel, fontsize=10)
    if xlim:   ax.set_xlim(xlim)
    if ylim:   ax.set_ylim(ylim)
    _clean(ax)


print('Plotting helpers OK')


Plotting helpers OK


## Session loading

Discovers all sessions for every subject listed in `GROUPS`, builds scratch workspaces,
runs `ddc.main()`, and computes all metrics inline. Output is a single `df` DataFrame.


In [ ]:
rows = []

for _group_name, _subject_ids in GROUPS.items():
    print(f'\n{"="*60}')
    print(f'Group: {_group_name}  ({len(_subject_ids)} animals)')
    print(f'{"="*60}')

    for _subject in _subject_ids:
        _attached = sorted(Path('/data').iterdir())
        _raws = [
            p for p in _attached
            if re.match(rf'single-plane-ophys_{_subject}_[\d-]+_\d{{2}}-\d{{2}}-\d{{2}}$',
                        p.name)
        ]
        if not _raws:
            print(f'  No assets for subject {_subject}')
            continue

        for _raw in _raws:
            _m = re.match(r'single-plane-ophys_(\d+)_([\d-]+)_', _raw.name)
            if not _m: continue
            _subj_id, _date = _m.group(1), _m.group(2)
            if DATE_FILTER and _date != DATE_FILTER: continue

            print(f'\n\u2500\u2500 {_subj_id}  {_date} ' + '\u2500'*max(1,38-len(_date)))
            try:
                _, _, _proc_root = find_pair_in_data(_subj_id, _date)
                _stem   = _best_bci_stem(_proc_root)
                _pophys = build_workspace(_subj_id, _date, _stem)

                with warnings.catch_warnings():
                    warnings.simplefilter('ignore', RuntimeWarning)
                    _data = ddc.main(str(_pophys) + '/')

                _dt = float(_data['dt_si'])
                _fs = 1.0 / _dt
                _cn = int(np.asarray(_data['conditioned_neuron'][0]).flat[0])

                # Behavioral arrays
                _tc  = _data['threshold_crossing_time']
                _si  = _data['SI_start_times']
                _tts = np.array([x[0] if x.size > 0 else np.nan for x in _tc])
                _sis = np.array([x[0] if x.size > 0 else np.nan for x in _si])
                _ttr_real = _tts - _sis
                _hit_mask = ~np.isnan(_ttr_real)
                _hit      = _hit_mask.astype(float)

                # Expected TTR per epoch
                _thresh = _data['BCI_thresholds']
                _sw     = epoch_switches(_thresh)
                _upr    = _thresh[1, _sw + 1]
                _lwr    = float(_thresh[0, _sw[0] + 1])
                _ep1end = int(_sw[1]) if len(_sw) > 1 else len(_ttr_real)
                _mu1    = float(np.nanmean(_ttr_real[:_ep1end]))
                _ttr_exp  = np.full(len(_ttr_real), np.nan)
                _exp_rate = np.full(len(_ttr_real), np.nan)
                for _si2 in range(len(_sw)):
                    _es = int(_sw[_si2])
                    _ee = int(_sw[_si2+1]) if _si2+1<len(_sw) else len(_ttr_real)
                    _al = ((_upr[0]-_lwr)/(_upr[_si2]-_lwr)
                           if (_upr[_si2]-_lwr)!=0 else 1.0)
                    _ttr_exp[_es:_ee]  = _mu1 / _al
                    _exp_rate[_es:_ee] = float(np.mean(_ttr_real[:_ep1end]/_al < 10))
                _ttr_ratio = _ttr_real / np.where(_ttr_exp>0, _ttr_exp, np.nan)

                # Trial window
                _nf = len(_ttr_ratio)
                _ts = min(TRIAL_START, _nf)
                _te = _nf - TRIAL_END_OFFSET
                if _te <= _ts:
                    print(f'  SKIP: {_nf} trials, window {_ts}:{_te} empty')
                    del _data; continue

                _tr   = _ttr_ratio[_ts:_te]
                _hit  = _hit[_ts:_te]
                _hm   = _hit_mask[_ts:_te]
                _hdif = moving_average(_hit, HIT_WINDOW) - _exp_rate[_ts:_te]
                _sw_w = sorted(set(np.clip(_sw-_ts, 0, len(_tr)).tolist()))
                if not _sw_w or _sw_w[0]!=0: _sw_w = [0]+_sw_w

                # CN activity
                _cn_act  = np.nanmean(_data['F'][:, _cn, :], axis=0)[_ts:_te]
                _cn_cont = np.array(_data['df_closedloop'][_cn, :], dtype=float)
                _bl_end  = max(4, _sw_w[1] if len(_sw_w)>1 else len(_cn_act))
                _cn_z    = zscore_to_baseline(_cn_act, 3, _bl_end)

                # Reward frames
                _rew_s = np.array([r[0] if r.size>0 else np.nan
                                   for r in _data['reward_time']])
                _tsa   = np.array(_data['trial_start'], dtype=float)
                _rew_abs = _tsa + np.where(np.isnan(_rew_s), np.nan, _rew_s)
                _rew_fr  = np.where(np.isnan(_rew_abs), np.nan,
                                    np.round(_rew_abs/_dt))
                _rew_fr_w = _rew_fr[_ts:_te]

                del _data

                # Peri-reward mean trace
                _peri_tr = peri_reward_trace(
                    _cn_cont, _rew_fr, _fs, BASELINE_SEC, PLOT_SEC)

                # Per-trial peri-reward (+-0.5 s)
                _hw = int(0.5*_fs); _nc = len(_cn_cont)
                _pt = np.full(len(_hit), np.nan)
                for _i2, _rf in enumerate(_rew_fr_w):
                    if np.isnan(_rf): continue
                    _rf = int(_rf)
                    if 0 <= _rf-_hw and _rf+_hw <= _nc:
                        _pt[_i2] = float(np.mean(_cn_cont[_rf-_hw:_rf+_hw]))
                _bl_pt = float(np.nanmean(_pt[:9]))
                _pt_n  = _pt - _bl_pt if not np.isnan(_bl_pt) else _pt.copy()
                del _cn_cont

                # Spike-based CN-population coupling
                _spks_path = _pophys / 'suite2p_BCI' / 'plane0' / 'spks.npy'
                _cn_pop_r  = np.nan
                if _spks_path.exists():
                    try:
                        _spks = np.load(_spks_path)
                        _ops2 = np.load(_pophys/'suite2p_BCI'/'plane0'/'ops.npy',
                                        allow_pickle=True).tolist()
                        _fpf2 = _ops2.get('frames_per_file', [])
                        _ics  = np.load(_pophys/'suite2p_BCI'/'plane0'/'iscell.npy',
                                        allow_pickle=True)
                        _cb   = (_ics[:,0] if _ics.ndim>1 else _ics).astype(bool)
                        _ncn  = _cb.copy(); _ncn[_cn] = False
                        _t_st = np.concatenate([[0], np.cumsum(_fpf2[:-1])]).astype(int)
                        _cn_s = np.array([
                            float(_spks[_cn,
                                  _t_st[_i3]:min(_t_st[_i3]+_fpf2[_i3],
                                                 _spks.shape[1])].sum())
                            for _i3 in range(len(_fpf2))])
                        _pp_s = np.array([
                            float(_spks[_ncn,
                                  _t_st[_i3]:min(_t_st[_i3]+_fpf2[_i3],
                                                 _spks.shape[1])].mean())
                            for _i3 in range(len(_fpf2))])
                        _cn_sw = _cn_s[_ts:_te]; _pp_sw = _pp_s[_ts:_te]
                        _vld3  = ~np.isnan(_cn_sw) & ~np.isnan(_pp_sw)
                        if _vld3.sum() >= 4:
                            _r3, _ = stats.pearsonr(_cn_sw[_vld3], _pp_sw[_vld3])
                            _cn_pop_r = float(_r3)
                    except Exception as _e3:
                        print(f'  spike coupling error: {_e3}')
                else:
                    print('  spks.npy not found in workspace')

                # Derived metrics
                _n       = len(_tr)
                _nhits   = int(_hm.sum())
                _third   = max(_n//3, 1)
                _quarter = max(_n//4, 1)
                _ep_edges = _sw_w + [_n]
                _ep_ttr   = [_safe_mean(_tr[int(_ep_edges[_e]):int(_ep_edges[_e+1])])
                             for _e in range(len(_ep_edges)-1)]
                _n_ep     = len(_ep_ttr)

                _pth  = _pt_n[_hm]; _xi = np.arange(len(_pth), dtype=float)
                _vld4 = ~np.isnan(_pth); _pt_r = np.nan
                if _vld4.sum() >= 4:
                    try: _pt_r = float(stats.pearsonr(_xi[_vld4], _pth[_vld4])[0])
                    except Exception: pass

                _lm = _cn_act[3:]; _nl = len(_lm)
                _lslope = _bws = _bwn = _bwm = np.nan
                if _nl >= WIN_SIZE:
                    _net = np.array([_lm[_j+WIN_SIZE-1]-_lm[_j]
                                     for _j in range(_nl-WIN_SIZE+1)])
                    _bi  = int(np.nanargmax(_net))
                    _win = _lm[_bi:_bi+WIN_SIZE]; _vl2 = ~np.isnan(_win)
                    if _vl2.sum() >= 4:
                        _res = stats.linregress(
                            np.arange(WIN_SIZE,dtype=float)[_vl2], _win[_vl2])
                        _lslope=float(_res.slope); _bws=int(_bi+3)
                        _bwn=float(_net[_bi]);     _bwm=float(np.nanmean(_win))

                _p12 = np.nan
                if _peri_tr is not None:
                    _tl = np.linspace(PLOT_SEC[0], PLOT_SEC[1], len(_peri_tr))
                    _p12 = _safe_mean(_peri_tr[(_tl>=1)&(_tl<=2)])

                rows.append(dict(
                    group              = _group_name,
                    subject            = _subj_id,
                    date               = _date,
                    n_trials           = _n,
                    n_hits             = _nhits,
                    n_epochs           = _n_ep,
                    fs_hz              = float(_fs),
                    hit_rate_overall   = _nhits/_n if _n>0 else np.nan,
                    hit_rate_first_10  = _safe_mean(_hm[:10]) if _n>=10 else np.nan,
                    hit_rate_last_10   = _safe_mean(_hm[-10:]) if _n>=10 else np.nan,
                    hit_rate_change_10 = (_safe_mean(_hm[-10:])-_safe_mean(_hm[:10])) if _n>=10 else np.nan,
                    hit_rate_first_3rd = _safe_mean(_hm[:_third]),
                    hit_rate_last_3rd  = _safe_mean(_hm[-_third:]),
                    mean_hit_diff      = _safe_mean(_hdif),
                    ttr_ratio_epoch1   = _ep_ttr[0]  if _ep_ttr else np.nan,
                    ttr_ratio_last_ep  = _ep_ttr[-1] if _ep_ttr else np.nan,
                    ttr_ratio_delta_ep = ((_ep_ttr[-1]-_ep_ttr[0])
                                          if len(_ep_ttr)>0
                                          and not np.isnan(_ep_ttr[0])
                                          and not np.isnan(_ep_ttr[-1])
                                          else np.nan),
                    ttr_ratio_median   = float(np.nanmedian(_tr[_hm])) if _nhits>0 else np.nan,
                    ttr_ratio_q1_med   = float(np.nanmedian(_tr[:_quarter])) if _quarter>0 else np.nan,
                    ttr_ratio_q4_med   = float(np.nanmedian(_tr[_n-_quarter:])) if _quarter>0 else np.nan,
                    ttr_ratio_q_delta  = (float(np.nanmedian(_tr[_n-_quarter:]))
                                         -float(np.nanmedian(_tr[:_quarter]))) if _quarter>0 else np.nan,
                    cn_directionality  = cn_directionality_metric(_cn_act[3:], CN_SMOOTH),
                    cn_act_z_mean      = _safe_mean(_cn_z),
                    cn_act_z_first_3rd = _safe_mean(_cn_z[:_third]),
                    cn_act_z_last_3rd  = _safe_mean(_cn_z[-_third:]),
                    cn_act_z_delta     = _safe_mean(_cn_z[-_third:])-_safe_mean(_cn_z[:_third]),
                    peri_reward_1_2s   = _p12,
                    mean_peri_trial    = _safe_mean(_pt_n),
                    peri_trial_corr    = _pt_r,
                    local_max_slope    = _lslope,
                    best_win_start     = _bws,
                    best_win_net_rise  = _bwn,
                    best_win_cn_mean   = _bwm,
                    cn_pop_spk_corr    = _cn_pop_r,
                ))
                print(f'  OK \u2014 {_n} trials | {_nhits} hits | {_n_ep} epochs | '
                      f'CN dir={cn_directionality_metric(_cn_act[3:],CN_SMOOTH):+.3f} | '
                      f'spk-pop r={_cn_pop_r:+.3f}')

            except Exception as _ex:
                print(f'  SKIPPED: {_ex}')
                traceback.print_exc()

# Build DataFrame
df_raw = pd.DataFrame(rows)
df = df_raw[
    (df_raw['n_hits']   >= MIN_HITS) &
    (df_raw['n_trials'] >= MIN_TRIALS)
].copy().reset_index(drop=True)
df['epochs_per_trial'] = df['n_epochs'] / df['n_trials']

print(f'\n{"="*60}')
print(f'Loaded {len(df_raw)} sessions \u2192 {len(df)} after exclusions')
for _g in GROUPS:
    _s = df[df['group']==_g]
    print(f'  {_g}: {len(_s)} sessions, {_s["subject"].nunique()} animals')
print(f'  cn_pop_spk_corr valid: {df["cn_pop_spk_corr"].notna().sum()}/{len(df)}')



Group: Ctrl  (4 animals)

── 820614  2026-05-13 ────────────────────────────


  workspace: grp_820614_2026-05-13_bci2  (130 TIFFs, 88460 frames, spks=True)
[bci] keeping 6 iscell==0 ROI(s) within 10 px of CN: [460, 612, 920, 926, 1858, 1888]
  SKIPPED: index 1934 is out of bounds for axis 1 with size 1934

── 820614  2026-05-15 ────────────────────────────
  SKIPPED: No processed asset for 820614/2026-05-15

── 820614  2026-05-18 ────────────────────────────


Traceback (most recent call last):
  File "<ipython-input-6-11d0ab2542d9>", line 33, in <module>
    _data = ddc.main(str(_pophys) + '/')
  File "/code/data_dict_create_module_bruker.py", line 144, in main
    data['F'], data['Fraw'], data['df_closedloop'], data['centroidX'], data['centroidY'] = create_BCI_F(Ftrace, ops, stat, pre, post)
  File "/code/data_dict_create_module_bruker.py", line 388, in create_BCI_F
    f = dff[:,ind]
IndexError: index 1934 is out of bounds for axis 1 with size 1934
Traceback (most recent call last):
  File "<ipython-input-6-11d0ab2542d9>", line 27, in <module>
    _, _, _proc_root = find_pair_in_data(_subj_id, _date)
  File "<ipython-input-3-f51d6076e9ee>", line 17, in find_pair_in_data
    if not procs: raise RuntimeError(f'No processed asset for {subject}/{date}')
RuntimeError: No processed asset for 820614/2026-05-15


  workspace: grp_820614_2026-05-18_bci  (87 TIFFs, 55157 frames, spks=True)
[bci] keeping 4 iscell==0 ROI(s) within 10 px of CN: [1250, 1417, 2146, 2739]
No suite2p_ch1/plane0 folder found — skipping ch1 data.
[bonsai] 88 Bonsai trials, 87 SI files, 87 matched
[MAIN] Non-photostim data saved as pickle: /scratch/grp_820614_2026-05-18_bci/pophys/data_main_scratch_grp_820614_2026-05-18_bci_BCI.npy
[MAIN] Non-photostim data saved as HDF5:   /scratch/grp_820614_2026-05-18_bci/pophys/data_main_scratch_grp_820614_2026-05-18_bci_BCI.h5
  OK — 67 trials | 67 hits | 5 epochs | CN dir=+0.186 | spk-pop r=+nan

── 820614  2026-05-20 ────────────────────────────
  workspace: grp_820614_2026-05-20_bci  (119 TIFFs, 77912 frames, spks=True)
[bci] keeping 2 iscell==0 ROI(s) within 10 px of CN: [22, 2239]
No suite2p_ch1/plane0 folder found — skipping ch1 data.
[bonsai] 119 Bonsai trials, 119 SI files, 119 matched
[MAIN] Non-photostim data saved as pickle: /scratch/grp_820614_2026-05-20_bci/pophys/data_ma

## Fig 1 \u00b7 Animal Performance

| Panel | Metric | Expected direction |
|-------|--------|-------------------|
| A | `hit_rate_overall` | Ctrl > LC-KO |
| B | `ttr_ratio_delta_ep` \u2014 threshold-normalized TTR change first\u2192last epoch | Ctrl < 0, LC-KO \u2248 0 |
| C | `epochs_per_trial` \u2014 threshold advances per trial | Ctrl > LC-KO |
| D | `ttr_ratio_q_delta` \u2014 Q4 vs Q1 TTR median, threshold-normalized | Ctrl < 0, LC-KO \u2248 0 |


In [ ]:
fig1, axes = plt.subplots(2, 2, figsize=(7, 6.5))

for _g in GROUPS:
    axes[0,0].scatter([], [], color=COLORS[_g], s=28, alpha=0.9,
                      label=_g, linewidths=0)
axes[0,0].legend(fontsize=9, frameon=False, loc='lower left', handletextpad=0.3)

strip_plot(axes[0,0], df, 'hit_rate_overall',
           ylabel='Hit rate')
_label(axes[0,0], 'A')

strip_plot(axes[0,1], df, 'ttr_ratio_delta_ep',
           ref_y=0.0,
           ylabel='TTR ratio \u0394\n(last \u2212 first epoch)')
_label(axes[0,1], 'B')

strip_plot(axes[1,0], df, 'epochs_per_trial',
           ylabel='Threshold advances\n/ trial')
_label(axes[1,0], 'C')

strip_plot(axes[1,1], df, 'ttr_ratio_q_delta',
           ref_y=0.0,
           ylabel='TTR ratio \u0394\n(Q4 \u2212 Q1)')
_label(axes[1,1], 'D')

fig1.suptitle('Fig 1  \u00b7  Animal Performance', fontsize=11)
fig1.tight_layout(rect=[0, 0, 1, 0.97])
fig1.savefig(OUT_DIR / 'fig1_performance.png')
plt.show()
print('Saved: fig1_performance.png')


Saved: fig1_performance.png


## Fig 2 \u00b7 CN Activity and Timing

| Panel | Metric |
|-------|-------|
| A | `cn_act_z_mean` \u2014 overall z-scored CN activity |
| B | Within-session CN trajectory: first \u2153 \u2192 last \u2153 (slope chart) |
| C | `peri_reward_1_2s` \u2014 post-reward CN activity 1\u20132 s |
| D | `best_win_cn_mean` \u2014 CN activity in peak learning window (LC-KO significantly higher, p=0.024 in prior session-level analysis) |


In [ ]:
fig2, axes = plt.subplots(2, 2, figsize=(7, 6.5))

strip_plot(axes[0,0], df, 'cn_act_z_mean',
           ylabel='CN activity\n(z-scored)')
_label(axes[0,0], 'A')

slope_chart(axes[0,1], df, 'cn_act_z_first_3rd', 'cn_act_z_last_3rd',
            ylabel='CN activity (z-scored)')
axes[0,1].set_title('Within-session trajectory', fontsize=9, pad=3)
_label(axes[0,1], 'B')

strip_plot(axes[1,0], df, 'peri_reward_1_2s',
           ref_y=0.0,
           ylabel='Post-reward CN\n(1\u20132 s)')
_label(axes[1,0], 'C')

strip_plot(axes[1,1], df, 'best_win_cn_mean',
           ylabel='CN \u2014 peak window')
_label(axes[1,1], 'D')

fig2.suptitle('Fig 2  \u00b7  CN Activity and Timing', fontsize=11)
fig2.tight_layout(rect=[0, 0, 1, 0.97])
fig2.savefig(OUT_DIR / 'fig2_cn_activity.png')
plt.show()
print('Saved: fig2_cn_activity.png')


Saved: fig2_cn_activity.png


## Fig 3 \u00b7 CN\u2013Behavior Relationships

Scatter plots (one dot = one session) coloured by group, with per-group OLS lines
and pooled Spearman \u03c1 annotated.

| Panel | x | y | Note |
|-------|---|---|------|
| A | `cn_act_z_mean` | `ttr_ratio_last_ep` | Strongest prior correlation (\u03c1=\u22120.73) |
| B | `peri_trial_corr` | `ttr_ratio_q_delta` | Peri-reward trend vs TTR improvement (\u03c1=\u22120.58) |
| C | `cn_act_z_delta` | `ttr_ratio_delta_ep` | Within-session CN ramp vs TTR change (multi-epoch only) |
| D | `cn_pop_spk_corr` | `ttr_ratio_q_delta` | CN\u2013population spike coupling vs behavioural improvement |


In [ ]:
fig3, axes = plt.subplots(2, 2, figsize=(7, 6.5))

for _g in GROUPS:
    axes[0,0].scatter([], [], color=COLORS[_g], s=28, alpha=0.9,
                      label=_g, linewidths=0)
axes[0,0].legend(fontsize=9, frameon=False, loc='upper right', handletextpad=0.3)

# A
scatter_corr(axes[0,0], df, 'cn_act_z_mean', 'ttr_ratio_last_ep',
             ref_y=1.0,
             xlabel='CN mean activity (z-scored)',
             ylabel='TTR ratio\n(last epoch)')
_label(axes[0,0], 'A')

# B
scatter_corr(axes[0,1], df, 'peri_trial_corr', 'ttr_ratio_q_delta',
             ref_x=0.0, ref_y=0.0,
             xlabel='Peri-trial CN corr.\n(r vs hit order)',
             ylabel='TTR ratio \u0394\n(Q4 \u2212 Q1)')
_label(axes[0,1], 'B')

# C — multi-epoch sessions only
_df_me = df[df['n_epochs'] > 1].copy()
scatter_corr(axes[1,0], _df_me, 'cn_act_z_delta', 'ttr_ratio_delta_ep',
             ref_x=0.0, ref_y=0.0,
             xlabel='CN \u0394 activity\n(last \u2212 first \u2153)',
             ylabel='TTR ratio \u0394\n(last \u2212 first epoch)')
_label(axes[1,0], 'C')

# D — CN-population spike coupling
if 'cn_pop_spk_corr' in df.columns and df['cn_pop_spk_corr'].notna().any():
    scatter_corr(axes[1,1], df, 'cn_pop_spk_corr', 'ttr_ratio_q_delta',
                 ref_x=0.0, ref_y=0.0,
                 xlabel='CN\u2013pop. spike coupling (r)',
                 ylabel='TTR ratio \u0394\n(Q4 \u2212 Q1)')
else:
    axes[1,1].text(0.5, 0.5,
                   'cn_pop_spk_corr not available\n(spks.npy missing)',
                   ha='center', va='center', transform=axes[1,1].transAxes,
                   fontsize=9, color='#999', style='italic')
    axes[1,1].set_xlabel('CN\u2013pop. spike coupling (r)', fontsize=10)
    axes[1,1].set_ylabel('TTR ratio \u0394\n(Q4 \u2212 Q1)', fontsize=10)
    axes[1,1].set_xticks([]); axes[1,1].set_yticks([])
    for _sp in ['top','right','left','bottom']:
        axes[1,1].spines[_sp].set_visible(False)
_label(axes[1,1], 'D')

fig3.suptitle('Fig 3  \u00b7  CN\u2013Behavior Relationships', fontsize=11)
fig3.tight_layout(rect=[0, 0, 1, 0.97])
fig3.savefig(OUT_DIR / 'fig3_cn_behavior.png')
plt.show()
print('Saved: fig3_cn_behavior.png')


Saved: fig3_cn_behavior.png


In [ ]:
_metrics = [
    ('hit_rate_overall',   'Hit rate'),
    ('ttr_ratio_delta_ep', 'TTR ratio \u0394 epoch (threshold-normalized)'),
    ('epochs_per_trial',   'Threshold advances / trial'),
    ('ttr_ratio_q_delta',  'TTR ratio \u0394 quartile (threshold-normalized)'),
    ('cn_act_z_mean',      'CN mean activity (z-scored)'),
    ('cn_act_z_delta',     'CN \u0394 activity (last \u2212 first \u2153)'),
    ('peri_reward_1_2s',   'Post-reward CN (1\u20132 s)'),
    ('best_win_cn_mean',   'CN \u2014 peak learning window'),
    ('peri_trial_corr',    'Peri-trial CN correlation'),
    ('cn_directionality',  'CN directionality'),
    ('cn_pop_spk_corr',    'CN\u2013population spike coupling'),
]
print(f'{"="*70}')
print('SUMMARY STATISTICS  (MWU on pooled session data)')
print('Sessions: ' + '  |  '.join(f'{g} n={len(df[df["group"]==g])}' for g in GROUPS))
print('NOTE: sessions within the same animal are not independent.')
print(f'{"="*70}')
print(f'  {"Metric":<42} '
      + '  '.join(f'{g:>8}' for g in GROUPS) + f'  {"p":>8}')
print(f'  {"-"*68}')
for _col, _lbl in _metrics:
    if _col not in df.columns: continue
    _meds = [df.loc[df['group']==g, _col].median() for g in GROUPS]
    _p, _ = group_mwu(df, _col)
    _sig  = ' *' if _p < 0.05 else '  '
    _ms   = '  '.join(f'{m:>8.3f}' for m in _meds)
    print(f'  {_lbl:<42} {_ms}  {_p:>8.3f}{_sig}')
print('\n  * p < 0.05 (MWU, pooled sessions, NOT corrected for pseudo-replication)')


SUMMARY STATISTICS  (MWU on pooled session data)
Sessions: Ctrl n=26  |  LC-KO n=22
NOTE: sessions within the same animal are not independent.
  Metric                                         Ctrl     LC-KO         p
  --------------------------------------------------------------------
  Hit rate                                      0.946     0.819     0.002 *
  TTR ratio Δ epoch (threshold-normalized)     -0.088     0.000     0.041 *
  Threshold advances / trial                    0.054     0.027     0.000 *
  TTR ratio Δ quartile (threshold-normalized)   -0.124    -0.063     0.127  
  CN mean activity (z-scored)                   0.081     0.017     0.501  
  CN Δ activity (last − first ⅓)                0.179     0.265     0.390  
  Post-reward CN (1–2 s)                        0.042     0.300     0.018 *
  CN — peak learning window                     1.309     1.708     0.148  
  Peri-trial CN correlation                    -0.003    -0.024     0.528  
  CN directionality        